In [1]:
import pandas as pd
import re
import instructor
from openai import AsyncOpenAI
import asyncio
from tqdm.asyncio import tqdm

In [2]:
aclient = instructor.patch(AsyncOpenAI())

In [41]:
model = "gpt-4o-mini"
numerical_regex = re.compile(r"^\d+[\.]?\s*")

def make_batches(titles, batch_size):
    return [titles[i:i+batch_size] for i in range(0, len(titles), batch_size)]

def rm_newlines(title):
    return re.sub(r"\n+", " ", title)

def make_input_content(titles):
    return "\n".join([f"{i+1}. "+ rm_newlines(title) for i, title in enumerate(titles)])

async def fetch_response(titles):
    content = make_input_content(titles)
    prompt = f"""You are a HR and language expert.
        You will be provided with a numerical list of job titles, and your task is to translate them into English.
        The output should be a numerical list of translated titles separated by newline."""
    messages = [
                    {
                        "role": "system",
                        "content": prompt
                    },
                    {
                        "role": "user",
                        "content": content
                    }
                ]

    try:
        response = await aclient.chat.completions.create(
            model=model,
            messages=messages
        )
        output = response.choices[0].message.content
        return output
    except Exception as e:
        print(f"Error processing input '{text}': {str(e)}")
        return None

def remove_numerical(title):
    return numerical_regex.sub("", title).strip()

def process_output_content(content):
    titles = content.split("\n")
    return [remove_numerical(title) for title in titles]

def get_titles(contents):
    titles = []
    for content in output_titles:
        if content:
            title_batch = process_output_content(content)
            if len(title_batch) == batch_size:
                titles += title_batch
            else:
                titles += [""] * batch_size
        else:
            titles += [""] * batch_size
    return titles

async def translate_titles(titles, batch_size: int=10):
    title_batches = make_batches(titles, batch_size)
    tasks = [fetch_response(titles) for titles in title_batches]
    responses = await tqdm.gather(*tasks)
    titles = get_titles(responses)
    return responses

In [42]:
df = pd.read_parquet("data/inputs/LT-1.parquet")
all_titles = df["job_title"].to_list()[:50]

In [43]:
output_titles = await translate_titles(all_titles, batch_size=5)

100%|███████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:04<00:00,  2.35it/s]


In [44]:
output_titles

['1. FAST FOOD BAKER | NO EXPERIENCE? WE WILL TRAIN YOU\n2. Salesperson - Consultant\n3. Administrator of the Operations Group of the Science and Innovation Center of the Agricultural Service\n4. MANAGER OF THE CUSTOMER SERVICE DEPARTMENT (0.5 FTE) (FIXED-TERM CONTRACT)\n5. Specialist of the Dainava Division of the South Lithuania Customer Service Center',
 '1. Accountant  \n2. Senior Accountant | Option to Work Part-Time from Home  \n3. Rail Baltica Business Analyst  \n4. Expert (Finance Sector) | Public and Private Partnership Department  \n5. Tax Consultant  ',
 '1. Business Analyst  \n2. Human Resources and Office Management Manager  \n3. Accountant  \n4. Female Accountant  \n5. Chief Accountant  ',
 '1. Chief Specialist of Operational Control (Public Servant)  \n2. Accountant  \n3. Specialist in the Financial Markets Operations Group  \n4. Accountant in the Financial Service  \n5. Senior Accountant / Accountant',
 '1. Senior Accountant  \n2. Accountant (3 days)  \n3. Senior Accoun

In [51]:
output_titles = [None, None] + output_titles

In [21]:
df = pd.read_parquet("data/inputs/LT-1.parquet")
titles = df["job_title"].to_list()[:10]
title_batchs = make_batches(titles, batch_size=2)

### Test module for different languages

In [7]:
import pandas as pd
from translate_titles import translate_titles

In [10]:
df_lt = pd.read_parquet("data/inputs/LT-1.parquet")
titles_lt = df_lt["job_title"].to_list()[:1000]

In [14]:
titles_en = await translate_titles(titles=titles, batch_size=50)

100%|███████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:24<00:00,  1.21s/it]


In [21]:
pd.DataFrame({
    "title": titles,
    "title_en": title_en
})

,title,title_en
0,GREITO MAISTO KEPĖJA (-AS) | NETURI PATIRTIES?...,FAST FOOD BAKER | NO EXPERIENCE? WE WILL TRAIN...
1,Pardavėja(-s) - konsultantė(-as,Salesperson - Consultant
2,Ūkio tarnybos Mokslo ir inovacijų centro ekspl...,Administrator of the Operations Group at the S...
3,KLIENTŲ APTARNAVIMO SKYRIAUS VADYBININKĖ (-AS)...,CUSTOMER SERVICE DEPARTMENT MANAGER (0.5 FTE) ...
4,Pietų Lietuvos Klientų aptarnavimo centro Dain...,Specialist at the Dainava Division of the Sout...
...,...,...
995,Tarptautinių mokestinės informacijos mainų spe...,International Tax Information Exchange Specialist
996,Vyr. teisininkas (-ė,Senior Lawyer
997,Teisininkė /- as,Lawyer
998,SPECIALISTAI (-ĖS) TEISINIŲ DOKUMENTŲ RENGIMUI,SPECIALISTS FOR LEGAL DOCUMENT PREPARATION


In [22]:
df_bd = pd.read_parquet("data/inputs/BD-1.parquet")
titles_bd = df_bd["job_title"].to_list()[:1000]

In [23]:
titles_en = await translate_titles(titles=titles_bd, batch_size=50)

100%|███████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:11<00:00,  1.77it/s]


In [24]:
pd.DataFrame({
    "title": titles_bd,
    "title_en": titles_en
})

,title,title_en
0,প্রভাষক,Lecturer
1,হিফয শিক্ষক ( আবাসিক ),Hafiz Teacher (Residential)
2,ইঞ্জিনিয়ার,Engineer
3,নায়েব,Deputy
4,অফিস সহায়ক,Office Assistant
...,...,...
995,Assistant Production Manager,Assistant Production Manager
996,Production Manager,Production Manager
997,রুম সার্ভিস বয়,Room Service Boy
998,Waiter,Waiter


In [32]:
pd.set_option('display.max_rows', 500)

In [27]:
df_id = pd.read_parquet("data/inputs/ID-1.parquet")
titles_id = df_id["job_title"].to_list()[:1000]

In [28]:
titles_en = await translate_titles(titles=titles_id, batch_size=50)

100%|███████████████████████████████████████████████████████████████████████████████████████████| 20/20 [00:40<00:00,  2.00s/it]


In [33]:
pd.DataFrame({
    "title": titles_id,
    "title_en": titles_en
})[:100]

,title,title_en
0,Micro Sales Officer - Surabaya & Sekitarnya,Micro Sales Officer - Surabaya & Surrounding A...
1,Hub Lead - Shopee Xpress (Banjarmasin),Hub Lead - Shopee Xpress (Banjarmasin)
2,Sales,Sales
3,Sales Admin,Sales Admin
4,Nail Terapis / Nail Art,Nail Therapist / Nail Art
5,Sales B2B / Commersial / Industri / Project (U...,B2B / Commercial / Industrial / Project Sales ...
6,Driver,Driver
7,Mekanik,Mechanic
8,Lash Terapis / Eyelash Extension,Lash Therapist / Eyelash Extension
9,Team Drafter,Team Drafter


In [30]:
titles_id

['Micro Sales Officer - Surabaya & Sekitarnya',
 'Hub Lead - Shopee Xpress (Banjarmasin)',
 'Sales',
 'Sales Admin',
 'Nail Terapis / Nail Art',
 'Sales B2B / Commersial / Industri / Project (URGENT)',
 'Driver',
 'Mekanik',
 'Lash Terapis / Eyelash Extension',
 'Team Drafter',
 'Reporter',
 'Driver',
 'Finance',
 'Drafter',
 'Teknisi UPS',
 'Staff Admin',
 'SALESMAN - MALANG',
 'Surveyor',
 'Sales Modern Trade Cabang medan',
 'Cook Helper',
 'Sales Toko',
 'Senior Sales Representatives - Jakarta/Tangerang',
 'Field Collector Area Manado',
 'Staff Administrasi & Packing Gudang',
 'Sales Cirebon',
 'Field Collector Penempatan Malang & Pasuruan',
 'Admin Toko Bahan Kue',
 'Cleaning Services',
 'Admin Staf',
 'Kepala Kendaraan',
 'SALES MARKETING',
 'DRIVER',
 'Operational Branch Support Staff (Pontianak Area)',
 'Staf Administrasi dan Akutansi',
 'Sales Project CCTV Area Serang & Cilegon',
 'BUILDING MAINTENANCE',
 'Marketing Verification Officer (JS - MVO) Sidoarjo',
 'SALES',
 'Retail 